# 04 — Crisis Attention Analysis

Compares Twitter attention volumes and engagement metrics across defined
crisis periods:

| Period label | Dates | Event |
|---|---|---|
| `pre_2014` | 2008-01 – 2013-11 | Baseline / pre-Euromaidan |
| `euromaidan` | 2013-11 – 2014-02 | Euromaidan protests |
| `crimea_donbas` | 2014-03 – 2021-12 | Annexation of Crimea & Donbas conflict |
| `invasion_2022` | 2022-02 – 2023-12 | Full-scale Russian invasion |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR = Path('../results/figures')
TABLES_DIR = Path('../results/tables')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Crisis period definitions (inclusive start, exclusive end)
CRISIS_PERIODS = {
    'pre_2014':     ('2008-01-01', '2013-11-21'),
    'euromaidan':   ('2013-11-21', '2014-02-28'),
    'crimea_donbas':('2014-03-01', '2022-02-24'),
    'invasion_2022':('2022-02-24', '2024-01-01'),
}

In [ ]:
clean_path = PROCESSED_DIR / 'tweets_clean.csv'
if clean_path.exists():
    tweets = pd.read_csv(clean_path, parse_dates=['created_at'])

    def assign_period(dt):
        for label, (start, end) in CRISIS_PERIODS.items():
            if pd.Timestamp(start) <= dt < pd.Timestamp(end):
                return label
        return 'other'

    tweets['crisis_period'] = tweets['created_at'].apply(assign_period)
    period_stats = tweets.groupby('crisis_period').agg(
        tweet_count=('id', 'count'),
        avg_retweets=('retweet_count', 'mean'),
        avg_likes=('favorite_count', 'mean'),
    ).reset_index()
    print(period_stats)
    period_stats.to_excel(TABLES_DIR / 'crisis_periods.xlsx', index=False)
    print('Table saved')
else:
    print('Run 02_preprocessing.ipynb first')